In [ ]:
import os
import glob
import json
import random
import time
import torch
import numpy as np
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, FuncFormatter, MaxNLocator
from PIL import Image

COLORS = {'clean': '#3d3d3a', 'ce': '#2a78d6', 'margin': '#eb6834', 'reference': '#8a8a85'}
TEMPERATURE_CMAP = plt.cm.viridis
TEMPERATURE_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X']

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans', 'font.size': 10, 'mathtext.fontset': 'dejavusans',
    'axes.labelsize': 10.5, 'axes.titlesize': 10.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'xtick.labelsize': 9.5, 'ytick.labelsize': 9.5, 'legend.fontsize': 9, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.8,
    'axes.edgecolor': '#3d3d3a', 'axes.labelcolor': '#1f1f1d', 'xtick.color': '#3d3d3a', 'ytick.color': '#3d3d3a',
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': '#e6e6e3', 'grid.linewidth': 0.6,
    'lines.linewidth': 1.8, 'lines.markersize': 6.5,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})


def legend_above(fig, handles, labels, ncol, title=None):
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=ncol, title=title,
               handlelength=2.2, columnspacing=1.6)


def save_figure(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(os.path.join(OUTPUT_DIR, f'{name}.{ext}'))


In [ ]:
def get_num_workers():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        print("Running on Kaggle environment")
        return min(8, os.cpu_count())
    elif 'COLAB_GPU' in os.environ:
        print("Running on Colab environment")
        return min(4, os.cpu_count())
    else:
        print("Running on a local or unknown environment")
        return 0 if os.name == 'nt' else min(2, os.cpu_count())

num_workers = get_num_workers()


class PathMNISTDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images    # (N, 28, 28, 3) uint8
        self.labels = labels    # (N,) int
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        img = Image.fromarray(self.images[index])
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[index])


IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if IS_KAGGLE:
    DATA_PATH = '/kaggle/input/datasets/mohamedihebhergli/medtech-pathmnist/pathmnist.npz'
    OUTPUT_DIR = '/kaggle/working'
    CHECKPOINT_SEARCH_DIRS = ['/kaggle/working', '/kaggle/input']
else:
    DATA_PATH = 'pathmnist.npz'
    OUTPUT_DIR = 'checkpoints'
    CHECKPOINT_SEARCH_DIRS = ['checkpoints']
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load official train / val / test splits from the .npz file
raw = np.load(DATA_PATH)
train_images = raw['train_images']   # (89996, 28, 28, 3) uint8
val_images   = raw['val_images']     # (10004, 28, 28, 3) uint8
test_images  = raw['test_images']    # (7180,  28, 28, 3) uint8
train_labels = raw['train_labels'].flatten()
val_labels   = raw['val_labels'].flatten()
test_labels  = raw['test_labels'].flatten()
print(f"Train: {len(train_images)}, validation: {len(val_images)}, test: {len(test_images)}")

# Per-channel mean and std of the training set (pixels in [0, 1])
_f = train_images.astype(np.float32) / 255.0
PATHMNIST_MEAN = tuple(_f.reshape(-1, 3).mean(axis=0).tolist())
PATHMNIST_STD  = tuple(_f.reshape(-1, 3).std(axis=0).tolist())
del _f
print(f"Mean: {tuple(f'{v:.4f}' for v in PATHMNIST_MEAN)}")
print(f"Std:  {tuple(f'{v:.4f}' for v in PATHMNIST_STD)}")

# 28x28 patches: random crop with 2px padding + horizontal flip.
# Vertical flip and rotation are common for histopathology but add robustness
# that is hard to separate from the effect of distillation; keeping augmentation
# light and consistent with the CIFAR-10 baseline.
transform_train = transforms.Compose([
    transforms.RandomCrop(28, padding=2),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(PATHMNIST_MEAN, PATHMNIST_STD),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(PATHMNIST_MEAN, PATHMNIST_STD),
])

trainset = PathMNISTDataset(train_images, train_labels, transform=transform_train)
valset   = PathMNISTDataset(val_images,   val_labels,   transform=transform_test)
testset  = PathMNISTDataset(test_images,  test_labels,  transform=transform_test)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True,  num_workers=num_workers)
valloader   = DataLoader(valset,   batch_size=128, shuffle=False, num_workers=num_workers)
testloader  = DataLoader(testset,  batch_size=128, shuffle=False, num_workers=num_workers)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("Warning: GPU not available. This will be slow for large datasets.")


In [ ]:
def initialize_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


In [ ]:
class PathMNISTModel(nn.Module):
    def __init__(self):
        super(PathMNISTModel, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # 28x28 -> 14x14 -> 7x7  =>  128 * 7 * 7 = 6272
        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 9),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x


In [ ]:
def evaluate_model(model, testloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


# Training at temperature T

For every temperature T in `TEMPERATURES`:

1. **Teacher**: trained on hard labels with softmax at temperature T, i.e. cross-entropy on `softmax(z / T)`.
2. **Student** (same architecture): trained on the teacher's soft labels `softmax(z_teacher / T)`, also at temperature T. Soft labels are computed on the fly for each (augmented) batch.
3. **Test time**: both models are evaluated at T = 1 (raw logits).

`teacher_T1` / `student_T1` are the no-distillation baseline (T = 1 means no temperature rescaling).

**Training protocol** (identical for all models): SGD, `EPOCHS` = 100, lr and momentum decayed 5× over training, paired seeds across temperatures.

**Checkpoints** are named `{role}_T{T}_e{epochs}.pth`; a `.meta.json` stores the full configuration and history. A checkpoint is reused only if its stored configuration matches the current one exactly.

In [ ]:
TEMPERATURES = [1, 5, 10, 20, 50]
EPOCHS = 100
EPOCHS_OVERRIDE = {}   # e.g. {50: 150} to train T=50 for more epochs
SEED_TEACHER = 0
SEED_STUDENT = 1

INITIAL_LR = 0.01
INITIAL_MOMENTUM = 0.9
LR_DECAY = 0.95
MOMENTUM_DECAY = 0.5
NUM_DECAYS = 5
MIN_MOMENTUM = 0.1

MAX_ACC_GAP = 2.0
MAX_LATE_GAIN = 0.5
MAX_LATE_DROP = 1.0

PROTOCOL = {
    'dataset': 'PathMNIST',
    'train_size': len(trainset),
    'val_size': len(valset),
    'split': 'medmnist-official',
    'augmentation': str(transform_train),
    'batch_size': trainloader.batch_size,
    'initial_lr': INITIAL_LR,
    'initial_momentum': INITIAL_MOMENTUM,
    'lr_decay': LR_DECAY,
    'momentum_decay': MOMENTUM_DECAY,
    'num_decays': NUM_DECAYS,
    'min_momentum': MIN_MOMENTUM,
    'loss_scaling': 'none',
    'gradient_clipping': 'none',
    'init': 'kaiming_uniform (initialize_weights)',
}


def epochs_for(T):
    return EPOCHS_OVERRIDE.get(T, EPOCHS)


def model_name(role, T):
    return f'{role}_T{T}_e{epochs_for(T)}'


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def save_json(obj, filename):
    with open(os.path.join(OUTPUT_DIR, filename), 'w') as f:
        json.dump(obj, f, indent=2)


In [ ]:
def run_training(model, loss_fn, epochs, tag):
    decay_step = max(1, epochs // NUM_DECAYS)
    optimizer = optim.SGD(model.parameters(), lr=INITIAL_LR, momentum=INITIAL_MOMENTUM)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=decay_step, gamma=LR_DECAY)
    history = {'epochs': epochs, 'train_loss': [], 'train_acc': [], 'val_acc': [], 'lr': [], 'momentum': []}

    for epoch in range(epochs):
        momentum = max(INITIAL_MOMENTUM * MOMENTUM_DECAY ** (epoch // decay_step), MIN_MOMENTUM)
        for param_group in optimizer.param_groups:
            param_group['momentum'] = momentum
        lr = optimizer.param_groups[0]['lr']

        epoch_start = time.time()
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(inputs)
            loss = loss_fn(logits, inputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()

        val_acc = evaluate_model(model, valloader)
        history['train_loss'].append(total_loss / len(trainloader))
        history['train_acc'].append(100 * correct / total)
        history['val_acc'].append(val_acc)
        history['lr'].append(lr)
        history['momentum'].append(momentum)
        print(f"[{tag}] Epoch {epoch+1}/{epochs}, LR: {lr:.5f}, Momentum: {momentum:.3f}, "
              f"Train Loss: {history['train_loss'][-1]:.4f}, Train Acc: {history['train_acc'][-1]:.2f}%, "
              f"Val Acc: {val_acc:.2f}%, Time: {time.time() - epoch_start:.0f}s")
    return history


def teacher_loss(T):
    def loss_fn(logits, inputs, labels):
        return F.cross_entropy(logits / T, labels)
    return loss_fn


def student_loss(teacher, T):
    teacher.eval()
    def loss_fn(logits, inputs, labels):
        with torch.no_grad():
            soft_labels = F.softmax(teacher(inputs) / T, dim=1)
        return F.kl_div(F.log_softmax(logits / T, dim=1), soft_labels, reduction='batchmean')
    return loss_fn


In [ ]:
def model_config(role, T, teacher_name=None):
    config = dict(PROTOCOL, role=role, T=T, epochs=epochs_for(T),
                  seed=SEED_TEACHER if role == 'teacher' else SEED_STUDENT)
    if teacher_name is not None:
        config['teacher'] = teacher_name
    return config


def find_model(name, config):
    candidates = []
    for d in CHECKPOINT_SEARCH_DIRS:
        candidates += sorted(glob.glob(os.path.join(d, '**', f'{name}.pth'), recursive=True))

    rejected = []
    for pth_path in candidates:
        meta_path = pth_path[:-len('.pth')] + '.meta.json'
        if not os.path.exists(meta_path):
            rejected.append(f'{pth_path}: no .meta.json file')
            continue
        with open(meta_path) as f:
            meta = json.load(f)
        if meta['config'] == config:
            return pth_path, meta
        diff = {k: (meta['config'].get(k), config.get(k))
                for k in set(meta['config']) | set(config) if meta['config'].get(k) != config.get(k)}
        rejected.append(f'{pth_path}: config differs (stored, current) = {diff}')

    if rejected:
        raise RuntimeError(f"Checkpoint(s) named {name}.pth exist but none matches the current configuration:\n"
                           + "\n".join(rejected)
                           + "\nRemove them from the inputs or change the configuration.")
    return None, None


histories = {}


def get_model(name, config, loss_fn_factory):
    path, meta = find_model(name, config)
    if path is not None:
        model = PathMNISTModel().to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        histories[name] = meta['history']
        print(f"Loaded {name} from {path}")
        import shutil
        for ext in ('.pth', '.meta.json'):
            src = path.replace('.pth', ext)
            dst = os.path.join(OUTPUT_DIR, name + ext)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
    else:
        set_seed(config['seed'])
        model = PathMNISTModel().to(device)
        model.apply(initialize_weights)
        history = run_training(model, loss_fn_factory(), config['epochs'], name)
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f'{name}.pth'))
        save_json({'config': config, 'history': history}, f'{name}.meta.json')
        histories[name] = history
        print(f"Saved {name} to {OUTPUT_DIR}")
    model.eval()
    return model


def get_teacher(T):
    name = model_name('teacher', T)
    return get_model(name, model_config('teacher', T), lambda: teacher_loss(T))


def get_student(T, teacher):
    name = model_name('student', T)
    config = model_config('student', T, teacher_name=model_name('teacher', T))
    return get_model(name, config, lambda: student_loss(teacher, T))


In [ ]:
teachers, students = {}, {}
for T in TEMPERATURES:
    print(f"========== T = {T} ==========")
    teachers[T] = get_teacher(T)
    students[T] = get_student(T, teachers[T])


In [ ]:
with torch.no_grad():
    inputs, _ = next(iter(valloader))
    inputs = inputs.to(device)
    for T in TEMPERATURES:
        z_teacher = teachers[T](inputs)
        z_student = students[T](inputs)
        soft = F.softmax(z_teacher / T, dim=1)
        print(f"T={T:>3}: teacher logit std {z_teacher.std().item():8.2f} | "
              f"student logit std {z_student.std().item():8.2f} | "
              f"mean max soft-label prob {soft.max(dim=1).values.mean().item():.3f} | "
              f"mean max prob of student at T=1 {F.softmax(z_student, dim=1).max(dim=1).values.mean().item():.4f}")


In [ ]:
TEMP_COLORS     = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7']
TEMP_LINESTYLES = ['-', '--', '-.', ':', '-']
TEMP_LINEWIDTHS = [1.6, 1.6, 1.6, 1.6, 2.2]


def marker_positions(n_epochs, i):
    step = max(1, n_epochs // 8)
    return (2 + i * 2, step)


def decollide_labels(items, min_gap=0.55):
    items = sorted(items, key=lambda x: x[0])
    positions = [items[0][0]]
    for y, color, text in items[1:]:
        positions.append(max(y, positions[-1] + min_gap))
    return [(pos, items[i][1], items[i][2]) for i, pos in enumerate(positions)]


ref_history = histories.get(model_name('teacher', 1))
ref_final = ref_history['val_acc'][-1] if ref_history else None

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.4), sharey=True)
for ax, role, panel_title in zip(axes, ['teacher', 'student'], ['(a) Teachers', '(b) Students']):
    if ref_final is not None:
        ax.axhline(ref_final, color='#aaaaaa', linewidth=1.0, linestyle='--', zorder=1)
    label_items = []
    for i, (T, color) in enumerate(zip(TEMPERATURES, TEMP_COLORS)):
        history = histories.get(model_name(role, T))
        if history is not None:
            n_epochs = len(history['val_acc'])
            ax.plot(range(1, n_epochs + 1), history['val_acc'],
                    color=color, linewidth=TEMP_LINEWIDTHS[i], label=f'{T}',
                    linestyle=TEMP_LINESTYLES[i],
                    marker=TEMPERATURE_MARKERS[i % len(TEMPERATURE_MARKERS)], markersize=5,
                    markevery=marker_positions(n_epochs, i))
            label_items.append((history['val_acc'][-1], color, f"{history['val_acc'][-1]:.1f}"))
    for y_label, color, text in decollide_labels(label_items):
        ax.annotate(text, xy=(EPOCHS, y_label), xytext=(3, 0), textcoords='offset points',
                    fontsize=7.5, color=color, va='center')
    ax.set_title(panel_title)
    ax.set_xlabel('Epoch')
    ax.set_xlim(0, EPOCHS + 10)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, steps=[1, 2, 5, 10]))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, steps=[1, 2, 5, 10]))
axes[0].set_ylabel('Validation accuracy (%)')
all_vals = [v for h in histories.values() if h is not None for v in h['val_acc']]
if all_vals:
    span = max(all_vals) - min(all_vals)
    margin = max(2.0, span * 0.2)
    axes[0].set_ylim(max(0, min(all_vals) - margin), min(100, max(all_vals) + margin))
if ref_final is not None:
    axes[0].annotate('baseline', xy=(5, ref_final), xytext=(0, 5),
                     textcoords='offset points', fontsize=7.5, color='#aaaaaa')
fig.tight_layout()
handles, labels = axes[0].get_legend_handles_labels()
legend_above(fig, handles, [f'T = {l}' for l in labels], ncol=len(labels))
save_figure(fig, 'fig_validation_curves')
plt.show()


def late_gain(val_acc):
    return float(np.mean(val_acc[-5:]) - np.mean(val_acc[-10:-5]))


def late_drop(val_acc):
    return float(np.max(val_acc) - np.mean(val_acc[-5:]))


reference_val_acc = evaluate_model(teachers[1], valloader)
print(f"Reference (no distillation) val accuracy: {reference_val_acc:.2f}% | flags: gap > {MAX_ACC_GAP}, "
      f"still improving (late gain > {MAX_LATE_GAIN}), declining (late drop > {MAX_LATE_DROP})\n")
print(f"{'model':>16} | {'train acc':>9} | {'val acc':>8} | {'gap':>6} | {'late gain':>9} | {'late drop':>9} | "
      f"{'test acc':>8} | check")
validation_check = {}
for T in TEMPERATURES:
    for role, model in [('teacher', teachers[T]), ('student', students[T])]:
        name = model_name(role, T)
        history = histories.get(name)
        val_acc = evaluate_model(model, valloader)
        test_acc = evaluate_model(model, testloader)
        gap = reference_val_acc - val_acc

        flags = []
        if gap > MAX_ACC_GAP:
            flags.append('gap')
        train_acc = history['train_acc'][-1] if history else float('nan')
        if history is None or len(history['val_acc']) < 10:
            gain = drop = float('nan')
            flags.append('no history')
        else:
            gain, drop = late_gain(history['val_acc']), late_drop(history['val_acc'])
            if gain > MAX_LATE_GAIN:
                flags.append('still improving')
            if drop > MAX_LATE_DROP:
                flags.append('declining')

        validation_check[name] = {'train_acc': train_acc, 'val_acc': val_acc, 'gap': gap, 'late_gain': gain,
                                  'late_drop': drop, 'test_acc': test_acc, 'flags': flags}
        status = 'ok' if not flags else 'NEEDS REVIEW (' + ', '.join(flags) + ')'
        print(f"{name:>16} | {train_acc:8.2f}% | {val_acc:7.2f}% | {gap:6.2f} | {gain:9.2f} | {drop:9.2f} | "
              f"{test_acc:7.2f}% | {status}")
save_json(validation_check, 'validation_check.json')

if any(v['flags'] for v in validation_check.values()):
    print("\nSome models need review. 'still improving' / 'gap': consider more epochs via EPOCHS_OVERRIDE.")


# Robustness Evaluation Against Adversarial Attacks

In [ ]:
class NormalizedModel(nn.Module):
    def __init__(self, model, mean=PATHMNIST_MEAN, std=PATHMNIST_STD):
        super().__init__()
        self.model = model
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return self.model((x - self.mean) / self.std)

testset_raw = PathMNISTDataset(test_images, test_labels, transform=transforms.ToTensor())
testloader_raw = DataLoader(testset_raw, batch_size=128, shuffle=False, num_workers=num_workers)


In [ ]:
def logit_margin(logits, labels):
    true_logit = logits.gather(1, labels[:, None]).squeeze(1)
    other_logits = logits.clone()
    other_logits.scatter_(1, labels[:, None], float('-inf'))
    return true_logit - other_logits.max(dim=1).values


def generate_pgd_adversarial_sample(model, inputs, labels, epsilon=8/255, alpha=2/255,
                                    num_iter=20, random_start=True, loss_type='ce', num_restarts=1):
    x_orig = inputs.detach()
    with torch.no_grad():
        fooled = model(x_orig).argmax(dim=1) != labels
    adv_inputs = x_orig.clone()

    for _ in range(num_restarts):
        x_adv = x_orig.clone()
        if random_start:
            x_adv = torch.clamp(x_adv + torch.empty_like(x_adv).uniform_(-epsilon, epsilon), 0, 1)

        for step in range(num_iter + 1):
            x_adv.requires_grad_(True)
            logits = model(x_adv)

            newly_fooled = (logits.detach().argmax(dim=1) != labels) & ~fooled
            adv_inputs[newly_fooled] = x_adv.detach()[newly_fooled]
            fooled |= newly_fooled
            if step == num_iter:
                break

            if loss_type == 'ce':
                loss = F.cross_entropy(logits, labels)
            elif loss_type == 'margin':
                loss = -logit_margin(logits, labels).sum()
            else:
                raise ValueError("loss_type must be 'ce' or 'margin'.")
            grad, = torch.autograd.grad(loss, x_adv)

            x_adv = x_adv.detach() + alpha * grad.sign()
            x_adv = torch.min(torch.max(x_adv, x_orig - epsilon), x_orig + epsilon)
            x_adv = torch.clamp(x_adv, 0, 1)

        adv_inputs[~fooled] = x_adv.detach()[~fooled]

    return adv_inputs.detach(), fooled


In [ ]:
def test_robustness(model, testloader, loss_type='ce', epsilon=8/255, alpha=2/255, num_iter=20,
                    num_restarts=1, seed=0):
    model.eval()
    set_seed(seed)
    n_total = n_clean_correct = n_robust = n_flipped = 0

    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        with torch.no_grad():
            clean_correct = model(inputs).argmax(dim=1) == labels
        _, fooled = generate_pgd_adversarial_sample(model, inputs, labels, epsilon, alpha, num_iter,
                                                    loss_type=loss_type, num_restarts=num_restarts)
        n_total += labels.size(0)
        n_clean_correct += clean_correct.sum().item()
        n_robust += (~fooled).sum().item()
        n_flipped += (clean_correct & fooled).sum().item()

    return {
        'clean_accuracy': 100 * n_clean_correct / n_total,
        'robust_accuracy': 100 * n_robust / n_total,
        'success_rate': 100 * n_flipped / max(n_clean_correct, 1),
    }


# PGD Robustness: No Distillation vs. Distilled Models

Both attacks use the same budget (L-infinity, epsilon = 8/255, step 2/255, 20 iterations, random start).
**CE-PGD** goes through the cross-entropy gradient and is expected to suffer from gradient masking at high T;
**Margin-PGD** operates on the logit margin and is unaffected by softmax saturation.
If distilled models appear robust under CE-PGD but not Margin-PGD, the robustness is gradient masking.

In [ ]:
ATTACKS = {'CE-PGD': 'ce', 'Margin-PGD': 'margin'}
PGD_SETTINGS = {'norm': 'Linf', 'epsilon': 8/255, 'alpha': 2/255, 'num_iter': 20,
                'num_restarts': 1, 'random_start': True, 'seed': 0}
PGD_RESTARTS = PGD_SETTINGS['num_restarts']

models_to_test = {'No distillation': teachers[1]}
models_to_test.update({f'T={T}': students[T] for T in TEMPERATURES})
model_names_map = {'No distillation': model_name('teacher', 1)}
model_names_map.update({f'T={T}': model_name('student', T) for T in TEMPERATURES})

results = {}
for name, model in models_to_test.items():
    wrapped_model = NormalizedModel(model).to(device).eval()
    results[name] = {'checkpoint': model_names_map[name]}
    for attack_name, loss_type in ATTACKS.items():
        r = test_robustness(wrapped_model, testloader_raw, loss_type=loss_type,
                            epsilon=PGD_SETTINGS['epsilon'], alpha=PGD_SETTINGS['alpha'],
                            num_iter=PGD_SETTINGS['num_iter'], num_restarts=PGD_SETTINGS['num_restarts'],
                            seed=PGD_SETTINGS['seed'])
        results[name]['clean_accuracy'] = r['clean_accuracy']
        results[name][attack_name] = {'robust_accuracy': r['robust_accuracy'], 'success_rate': r['success_rate']}
        print(f"{name:>16} | {attack_name:>10}: clean {r['clean_accuracy']:.2f}%, "
              f"robust {r['robust_accuracy']:.2f}%, "
              f"success rate {r['success_rate']:.2f}% (of correctly classified samples)")

save_json({'settings': PGD_SETTINGS, 'results': results}, 'pgd_results.json')


In [ ]:
model_keys = ['No distillation'] + [f'T={T}' for T in TEMPERATURES]
x = np.arange(len(model_keys))
x_labels = ['No\ndistill.'] + [str(T) for T in TEMPERATURES]

DODGE = {'Clean': 0.0, 'CE-PGD': -0.09, 'Margin-PGD': 0.09}


def plot_vs_temperature(ax, values, color, marker, label):
    xs = x + DODGE[label]
    ax.plot(xs[1:], values[1:], color=color, marker=marker, label=label, zorder=3)
    ax.plot(xs[:1], values[:1], linestyle='none', marker=marker, color=color,
            markerfacecolor='white', markeredgewidth=1.5, zorder=3)


def format_temperature_axis(ax):
    ax.axvline(0.5, color=COLORS['reference'], linewidth=0.8, linestyle=':')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels)
    ax.set_xlim(-0.4, len(x) - 0.6)
    ax.set_xlabel('Distillation temperature T')


hollow_handle = Line2D([], [], linestyle='none', marker='o', color=COLORS['reference'],
                       markerfacecolor='white', markeredgewidth=1.5, label='No distillation (hollow markers)')

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1))
ax = axes[0]
plot_vs_temperature(ax, [results[k]['clean_accuracy'] for k in model_keys], COLORS['clean'], 'o', 'Clean')
plot_vs_temperature(ax, [results[k]['CE-PGD']['robust_accuracy'] for k in model_keys], COLORS['ce'], 's', 'CE-PGD')
plot_vs_temperature(ax, [results[k]['Margin-PGD']['robust_accuracy'] for k in model_keys], COLORS['margin'], '^', 'Margin-PGD')
ax.set_ylabel('Accuracy (%)')
ax.set_title('(a) Clean and robust accuracy')
ax.set_ylim(-4, 104)
format_temperature_axis(ax)

ax = axes[1]
plot_vs_temperature(ax, [results[k]['CE-PGD']['success_rate'] for k in model_keys], COLORS['ce'], 's', 'CE-PGD')
plot_vs_temperature(ax, [results[k]['Margin-PGD']['success_rate'] for k in model_keys], COLORS['margin'], '^', 'Margin-PGD')
ax.set_ylabel('Success rate (%)\n(clean-correct samples)')
ax.set_title('(b) Attack success rate')
ax.set_ylim(-4, 104)
format_temperature_axis(ax)

fig.tight_layout()
handles, labels = axes[0].get_legend_handles_labels()
legend_above(fig, handles + [hollow_handle], labels + [hollow_handle.get_label()], ncol=4)
save_figure(fig, 'fig_accuracy_vs_temperature')
plt.show()
print(f"PGD: L-inf, epsilon = {PGD_SETTINGS['epsilon']:.4f} (8/255), step = {PGD_SETTINGS['alpha']:.4f} (2/255), "
      f"{PGD_SETTINGS['num_iter']} iterations, {PGD_SETTINGS['num_restarts']} restart(s), random start.")


# Average Gradient Norm

For each model we compute per-sample input gradients in [0,1] pixel space:

* **Cross-entropy loss** `CE(z(x), y)`: goes through the softmax; vanishes when saturation is present.
* **Logit margin** `m(x) = z_y(x) - max_{j != y} z_j(x)`: does not go through the softmax.

A **scale-invariant** diagnostic is `d_inf = m(x) / ||grad m(x)||_1` — the linearized distance to the decision boundary in the L-infinity metric (compare with epsilon = 8/255). "Small CE gradient, large margin gradient" is the fingerprint of gradient masking.

In [ ]:
def input_gradient_stats(model, loader):
    model.eval()
    out = {k: [] for k in ['ce_l2', 'ce_l1', 'margin_l2', 'margin_l1', 'margin', 'correct']}

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs.requires_grad_(True)
        logits = model(inputs)
        margins = logit_margin(logits, labels)

        objectives = {'ce': F.cross_entropy(logits, labels, reduction='sum'), 'margin': margins.sum()}
        for key, objective in objectives.items():
            grad, = torch.autograd.grad(objective, inputs, retain_graph=(key == 'ce'))
            grad = grad.flatten(start_dim=1)
            out[f'{key}_l2'].append(grad.norm(p=2, dim=1).cpu())
            out[f'{key}_l1'].append(grad.norm(p=1, dim=1).cpu())
        out['margin'].append(margins.detach().cpu())
        out['correct'].append((logits.detach().argmax(dim=1) == labels).cpu())

    return {k: torch.cat(v).numpy() for k, v in out.items()}


def summarize_gradient_stats(s, epsilon=8/255):
    summary = {}
    for subset, mask in [('correct', s['correct']), ('wrong', ~s['correct'])]:
        summary[subset] = {'n': int(mask.sum())}
        for key in ['ce', 'margin']:
            l2, l1 = s[f'{key}_l2'][mask], s[f'{key}_l1'][mask]
            summary[subset][key] = {
                'mean_l2': float(l2.mean()) if mask.any() else float('nan'),
                'median_l2': float(np.median(l2)) if mask.any() else float('nan'),
                'mean_l1': float(l1.mean()) if mask.any() else float('nan'),
                'median_l1': float(np.median(l1)) if mask.any() else float('nan'),
                'zero_frac': float((l2 == 0).mean()) if mask.any() else float('nan'),
            }

    mask = s['correct']
    m, l1, l2 = s['margin'][mask], s['margin_l1'][mask], s['margin_l2'][mask]
    with np.errstate(divide='ignore'):
        d_inf = np.where(l1 > 0, m / l1, np.inf)
        d_2   = np.where(l2 > 0, m / l2, np.inf)
    summary['distance'] = {
        'median_d_inf': float(np.median(d_inf)) if mask.any() else float('nan'),
        'median_d_2':   float(np.median(d_2))   if mask.any() else float('nan'),
        'frac_d_inf_below_eps': float((d_inf < epsilon).mean()) if mask.any() else float('nan'),
        'n_infinite': int(np.isinf(d_inf).sum()),
    }
    return summary


In [ ]:
grad_results, grad_per_sample = {}, {}
for name, model in models_to_test.items():
    wrapped_model = NormalizedModel(model).to(device).eval()
    per_sample = input_gradient_stats(wrapped_model, testloader_raw)
    summary = summarize_gradient_stats(per_sample, epsilon=PGD_SETTINGS['epsilon'])
    grad_results[name] = summary
    grad_per_sample[name] = per_sample

    c, w, d = summary['correct'], summary['wrong'], summary['distance']
    print(f"{name:>16} | correct (n={c['n']}): CE median L2 {c['ce']['median_l2']:.3e}, "
          f"zero {100 * c['ce']['zero_frac']:.1f}% | margin median L2 {c['margin']['median_l2']:.3e}")
    print(f"{'':>16} | wrong   (n={w['n']}): CE median L2 {w['ce']['median_l2']:.3e}, "
          f"zero {100 * w['ce']['zero_frac']:.1f}% | margin median L2 {w['margin']['median_l2']:.3e}")
    print(f"{'':>16} | linearized distance (correct): median d_inf {d['median_d_inf']:.4f} "
          f"(eps = {PGD_SETTINGS['epsilon']:.4f}), below eps {100 * d['frac_d_inf_below_eps']:.1f}%, "
          f"median d_2 {d['median_d_2']:.4f}")

save_json({'checkpoints': model_names_map, 'results': grad_results}, 'gradient_norms.json')
np.savez(os.path.join(OUTPUT_DIR, 'gradient_norms_per_sample.npz'),
         **{f'{name}|{key}': values for name, d in grad_per_sample.items() for key, values in d.items()})


In [ ]:
def median_iqr(values):
    positive = values[np.isfinite(values) & (values > 0)]
    zero_frac = 1 - len(positive) / max(len(values), 1)
    if len(positive) == 0:
        return np.nan, 0.0, 0.0, zero_frac
    q1, med, q3 = np.percentile(positive, [25, 50, 75])
    return med, med - q1, q3 - med, zero_frac


def linearized_distance(s):
    with np.errstate(divide='ignore', invalid='ignore'):
        return s['margin'] / s['margin_l1']


quantities = [
    ('(a) Cross-entropy gradient', r'$\|\nabla_x \mathrm{CE}\|_2$', lambda s: s['ce_l2']),
    ('(b) Logit-margin gradient',  r'$\|\nabla_x m\|_2$',            lambda s: s['margin_l2']),
    ('(c) Linearized distance',    r'$d_\infty = m \,/\, \|\nabla_x m\|_1$', linearized_distance),
]

fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.1))
for ax, (title, ylabel, get) in zip(axes, quantities):
    stats = [median_iqr(get(grad_per_sample[k])[grad_per_sample[k]['correct']]) for k in model_keys]
    med = np.array([s[0] for s in stats])
    err = np.array([[s[1] for s in stats], [s[2] for s in stats]])
    ax.errorbar(x[1:], med[1:], yerr=err[:, 1:], color=COLORS['ce'], marker='o', capsize=3, elinewidth=1,
                label='Distilled', zorder=3)
    ax.errorbar(x[:1], med[:1], yerr=err[:, :1], color=COLORS['ce'], marker='o', markerfacecolor='white',
                markeredgewidth=1.5, capsize=3, elinewidth=1, linestyle='none', label='No distillation', zorder=3)
    for xi, (m, lo, hi, zero_frac) in zip(x, stats):
        if zero_frac > 0 and np.isfinite(m):
            ax.annotate(f'{100 * zero_frac:.0f}% = 0', (xi, m + hi), textcoords='offset points', xytext=(0, 4),
                        ha='center', fontsize=8, color=COLORS['clean'])
    if title.startswith('(c)'):
        ax.axhline(PGD_SETTINGS['epsilon'], color=COLORS['clean'], linestyle='--', linewidth=1)
        ax.annotate(r'$\epsilon = 8/255$', (len(x) - 0.6, PGD_SETTINGS['epsilon']), textcoords='offset points',
                    xytext=(-2, -12), ha='right', fontsize=8.5, color=COLORS['clean'])
    ax.set_yscale('log')
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs='auto'))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())
    ax.grid(True, axis='y', which='major')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    format_temperature_axis(ax)
fig.tight_layout()
handles, labels = axes[0].get_legend_handles_labels()
legend_above(fig, handles, ['Distilled (median, IQR)', 'No distillation (median, IQR)'], ncol=2)
save_figure(fig, 'fig_gradients_vs_temperature')
plt.show()
